# 提示词迁移指南
GPT-4.1 等较新的模型在性能和指令遵循方面都是同类最佳。随着模型越来越聪明，我们需要不断调整那些最初为适应早期模型局限而编写的提示词，确保它们对新一代模型依然有效且清晰。

GPT‑4.1 这类模型非常擅长严格遵循指令，但这种精确性也意味着它可能会**按字面意思**理解不清晰或措辞不当的指令，从而导致意外或错误的结果。为了充分发挥 GPT‑4.1 的潜力，必须对提示词进行打磨，确保每条指令都明确、无歧义，并与你期望的结果保持一致。

---

不清晰指令的示例：

- 含糊：

> “不要包含无关信息。”

问题：如果没有明确定义，GPT-4.1 可能难以判断什么是“无关的”。这可能导致它因过于谨慎的理解而遗漏重要细节，或无意中包含过多细节。

- 改进后：

> “只包含与主题（X）直接相关的事实。排除个人轶事、无关的历史背景或旁支讨论。”

---

**目标**：本交互式 notebook 帮助你按照最佳实践，将现有提示词（为其他模型编写）改进为清晰、无歧义且针对 GPT‑4.1 优化的提示词。

**工作流概览**  
本 notebook 采用以下方法：

- [第 1 步：输入你的原始提示词](#第-1-步输入你的原始提示词)  
- [第 2 步：找出提示词中的所有指令](#第-2-步找出提示词中的所有指令)  
- [第 3 步：让 GPT-4.1 *评审*提示词](#第-3-步让-gpt-41-评审提示词)  
- [第 4 步：自动生成修订后的 system 提示词](#第-4-步自动生成修订后的-system-提示词)  
- [第 5 步：评估与迭代](#第-5-步评估与迭代)  
- [第 6 步：（可选）自动应用 GPT-4.1 最佳实践](#第-6-步可选自动应用-gpt-41-最佳实践)

**前提条件**
- `openai` Python 包和 `OPENAI_API_KEY`

In [1]:
# !pip install openai pydantic tiktoken

In [1]:
# Imports & API connection
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Any, Dict, Iterable, List, Optional
import tiktoken
import html
from html import escape  
import difflib
import sys

from IPython.display import display, HTML

try:
    from IPython.display import HTML, display
    _IN_IPYTHON = True
except ImportError:
    _IN_IPYTHON = False


client = OpenAI()

MODEL = "gpt-4.1"

下面是几个辅助函数，方便我们查看对提示词的分析和修改。

In [3]:
_COLORS = {
    '+': ("#d2f5d6", "#22863a"),   # additions  (green)
    '-': ("#f8d7da", "#b31d28"),   # deletions  (red)
    '@': (None,      "#6f42c1"),   # hunk header (purple)
}

def _css(**rules: str) -> str:
    """Convert kwargs to a CSS string (snake_case → kebab-case)."""
    return ";".join(f"{k.replace('_', '-')}: {v}" for k, v in rules.items())

def _render(html_str: str) -> None:
    """Render inside Jupyter if available, else print to stdout."""
    try:
        display  # type: ignore[name-defined]
        from IPython.display import HTML  # noqa: WPS433
        display(HTML(html_str))
    except NameError:
        print(html_str, flush=True)

# ---------- diff helpers ------------------------------------------------------

def _style(line: str) -> str:
    """Wrap a diff line in a <span> with optional colors."""
    bg, fg = _COLORS.get(line[:1], (None, None))
    css = ";".join(s for s in (f"background:{bg}" if bg else "",
                               f"color:{fg}" if fg else "") if s)
    return f'<span style="{css}">{html.escape(line)}</span>'

def _wrap(lines: Iterable[str]) -> str:
    body = "<br>".join(lines)
    return (
        "<details>"
        "<summary>🕵️‍♂️ Critique & Diff (click to expand)</summary>"
        f'<div style="font-family:monospace;white-space:pre;">{body}</div>'
        "</details>"
    )

def show_critique_and_diff(old: str, new: str) -> str:
    """Display & return a GitHub-style HTML diff between *old* and *new*."""
    diff = difflib.unified_diff(old.splitlines(), new.splitlines(),
                                fromfile="old", tofile="new", lineterm="")
    html_block = _wrap(map(_style, diff))
    _render(html_block)
    return html_block

# ---------- “card” helpers ----------------------------------------------------

CARD    = _css(background="#f8f9fa", border_radius="8px", padding="18px 22px",
               margin_bottom="18px", border="1px solid #e0e0e0",
               box_shadow="0 1px 4px #0001")
TITLE   = _css(font_weight="600", font_size="1.1em", color="#2d3748",
               margin_bottom="6px")
LABEL   = _css(color="#718096", font_size="0.95em", font_weight="500",
               margin_right="6px")
EXTRACT = _css(font_family="monospace", background="#f1f5f9", padding="7px 10px",
               border_radius="5px", display="block", margin_top="3px",
               white_space="pre-wrap", color="#1a202c")

def display_cards(
    items: Iterable[Any],
    *,
    title_attr: str,
    field_labels: Optional[Dict[str, str]] = None,
    card_title_prefix: str = "Item",
) -> None:
    """Render objects as HTML “cards” (or plaintext when not in IPython)."""
    items = list(items)
    if not items:
        _render("<em>No data to display.</em>")
        return

    # auto-derive field labels if none supplied
    if field_labels is None:
        sample = items[0]
        field_labels = {
            a: a.replace("_", " ").title()
            for a in dir(sample)
            if not a.startswith("_")
            and not callable(getattr(sample, a))
            and a != title_attr
        }

    cards = []
    for idx, obj in enumerate(items, 1):
        title_html = html.escape(str(getattr(obj, title_attr, "<missing title>")))
        rows = [f'<div style="{TITLE}">{card_title_prefix} {idx}: {title_html}</div>']

        for attr, label in field_labels.items():
            value = getattr(obj, attr, None)
            if value is None:
                continue
            rows.append(
                f'<div><span style="{LABEL}">{html.escape(label)}:</span>'
                f'<span style="{EXTRACT}">{html.escape(str(value))}</span></div>'
            )

        cards.append(f'<div style="{CARD}">{"".join(rows)}</div>')

    _render("\n".join(cards))

## 第 1 步：输入你的原始提示词
首先，将你现有的提示词清晰地放在三引号（`"""`）之间。该提示词将作为改进的基线。

在本示例中，我们将使用以下[论文](https://arxiv.org/pdf/2306.05685)中提供的 LLM-as-a-Judge 系统提示词。

In [4]:
original_prompt = """
[System]
Please act as an impartial judge and evaluate the quality of the responses provided by two
AI assistants to the user question displayed below. You should choose the assistant that
follows the user’s instructions and answers the user’s question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.

[User Question]
{question}

[The Start of Assistant A’s Answer]
{answer_a}
[The End of Assistant A’s Answer]

[The Start of Assistant B’s Answer]
{answer_b}
[The End of Assistant B’s Answer]
"""

encoding = tiktoken.encoding_for_model("gpt-4")
num_tokens = len(encoding.encode(original_prompt))
print("Original prompt length:", num_tokens, "tokens")

Original prompt length: 243 tokens


## 第 2 步：找出提示词中的所有指令
在本节中，我们将提取 LLM 在系统提示词中识别出的每一条指令（INSTRUCTION）。这样你就可以检查该列表，找出不应作为指令的语句，并澄清任何含糊的指令。

请仔细检查并确认列出的每条指令都准确且有必要保留。

In [5]:
class Instruction(BaseModel):
    instruction_title: str = Field(description="A 2-8 word title of the instruction that the LLM has to follow.")
    extracted_instruction: str = Field(description="The exact text that was extracted from the system prompt that the instruction is derived from.")

class InstructionList(BaseModel):
    instructions: list[Instruction] = Field(description="A list of instructions and their corresponding extracted text that the LLM has to follow.")


EXTRACT_INSTRUCTIONS_SYSTEM_PROMPT = """
## Role & Objective
You are an **Instruction-Extraction Assistant**.  
Your job is to read a System Prompt provided by the user and distill the **mandatory instructions** the target LLM must obey.

## Instructions
1. **Identify Mandatory Instructions**  
   • Locate every instruction in the System Prompt that the LLM is explicitly required to follow.  
   • Ignore suggestions, best-practice tips, or optional guidance.

2. **Generate Rules**  
   • Re-express each mandatory instruction as a clear, concise rule.
   • Provide the extracted text that the instruction is derived from.
   • Each rule must be standalone and imperative.

## Output Format
Return a json object with a list of instructions which contains an instruction_title and their corresponding extracted text that the LLM has to follow. Do not include any other text or comments.

## Constraints
- Include **only** rules that the System Prompt explicitly enforces.  
- Omit any guidance that is merely encouraged, implied, or optional.  
"""

response = client.responses.parse(
    model=MODEL,
    input="SYSTEM_PROMPT TO ANALYZE: " + original_prompt,
    instructions=EXTRACT_INSTRUCTIONS_SYSTEM_PROMPT,
    temperature=0.0,
    text_format=InstructionList,
)

instructions_list = response.output_parsed

In [6]:
display_cards(
    instructions_list.instructions,
    title_attr="instruction_title",
    field_labels={"extracted_instruction": "Extracted Text"},
    card_title_prefix="Instruction"
)

检查模型将提示词中的哪些部分识别为指令很有帮助。指令是我们用自然语言“编程”模型的方式，因此确保它们清晰、精确且正确至关重要。

## 第 3 步：让 GPT-4.1 *评审*提示词
接下来，GPT‑4.1 将亲自评审原始提示词，专门找出可能导致混淆或错误的地方：

- 歧义：可能有多种解读的措辞。

- 缺少定义：未明确定义的标签或术语，可能导致模型去推断或猜测其含义。

- 指令冲突：相互矛盾或重叠的规则或条件。

- 缺少上下文或假设：未明确提供的必要信息或上下文。

评审输出将条理清晰，突出具体问题，并给出可操作的改进建议。

模型非常擅长**识别提示词中它们认为含糊或令人困惑的部分**。通过解决这些问题，我们可以对指令进行设计，使其对模型而言更清晰、更有效。

In [18]:
class CritiqueIssue(BaseModel):
    issue: str
    snippet: str
    explanation: str
    suggestion: str

class CritiqueIssues(BaseModel):
    issues: List[CritiqueIssue] = Field(..., min_length=1, max_length=6)
    
CRITIQUE_SYSTEM_PROMPT = """
## Role & Objective  
You are a **Prompt-Critique Assistant**.
Examine a user-supplied LLM prompt (targeting GPT-4.1 or compatible) and surface any weaknesses.

## Instructions
Check for the following issues:
- Ambiguity: Could any wording be interpreted in more than one way?
- Lacking Definitions: Are there any class labels, terms, or concepts that are not defined that might be misinterpreted by an LLM?
- Conflicting, missing, or vague instructions: Are directions incomplete or contradictory?
- Unstated assumptions: Does the prompt assume the model has to be able to do something that is not explicitly stated?

## Do **NOT** list issues of the following types:
- Invent new instructions, tool calls, or external information. You do not know what tools need to be added that are missing.
- Issues that you are not sure about.

## Output Format  
Return a JSON **array** (not an object) with 1-6 items, each following this schema:

```json
{
  "issue":      "<1-6 word label>",
  "snippet":    "<≤50-word excerpt>",
  "explanation":"<Why it matters>",
  "suggestion": "<Actionable fix>"
}
Return a JSON array of these objects. If the prompt is already clear, complete, and effective, return an empty list: `[]`.
"""

CRITIQUE_USER_PROMPT = f"""
Evaluate the following prompt for clarity, completeness, and effectiveness:
###
{original_prompt}
###
Return your critique using the specified JSON format only.
"""



In [19]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": CRITIQUE_SYSTEM_PROMPT},
        {"role": "user", "content": CRITIQUE_USER_PROMPT},
    ],
    temperature=0.0,
    text_format=CritiqueIssues,
)

critique = response.output_parsed

In [20]:
display_cards(
    critique.issues,
    title_attr="issue",
    field_labels={
        "snippet": "Snippet",
        "explanation": "Explanation",
        "suggestion": "Suggestion"
    },
    card_title_prefix="Issue"
)

In [21]:
# Create a string of the issues
issues_str = "\n".join(
    f"Issue: {issue.issue}\nSnippet: {issue.snippet}\nExplanation: {issue.explanation}\nSuggestion: {issue.suggestion}\n"
    for issue in critique.issues
)

print(issues_str)

Issue: Ambiguous evaluation criteria
Snippet: consider factors such as the helpfulness, relevance, accuracy, depth, creativity, and level of detail
Explanation: The prompt lists several evaluation factors but does not define them or explain how to weigh them. This could lead to inconsistent or subjective judgments.
Suggestion: Provide clear definitions for each criterion and specify if any should be prioritized over others.

Issue: Unclear handling of ties
Snippet: "[[C]]" for a tie
Explanation: The prompt allows for a tie verdict but does not specify under what circumstances a tie is appropriate, which may lead to inconsistent use.
Suggestion: Clarify when a tie should be chosen, e.g., if both responses are equally strong across all criteria.

Issue: Potential ambiguity in 'objectivity'
Snippet: Be as objective as possible.
Explanation: The prompt asks for objectivity but does not specify what constitutes objectivity in this context, especially given the subjective nature of some crit

检查问题列表：
- 如果你对这些问题没有异议，请继续进行下一步（第 4 步）。 
- 如果你认为某些问题不相关，请将上面的文本复制到下一个单元格中并删除这些问题。在本例中，三个问题都很合理，因此我们跳过这一步。

In [22]:
# issues_str = """
# PLACEHOLDER FOR ISSUES YOU WANT TO CORRECT, DO NOT RUN THIS CELL UNLESS YOU HAVE COPY-PASTED THE ISSUES FROM ABOVE
# """

## 第 4 步：自动生成修订后的 *system* 提示词
现在我们将评审意见反馈给 GPT‑4.1，让它生成原始提示词的改进版本，可直接用作 `system` 角色消息。

In [23]:
REVISE_SYSTEM_PROMPT = """
## Role & Objective  
Revise the user’s original prompt to resolve most of the listed issues, while preserving the original wording and structure as much as possible.

## Instructions
1. Carefully review the original prompt and the list of issues.
2. Apply targeted edits directly addressing the listed issues. The edits should be as minimal as possible while still addressing the issue.
3. Do not introduce new content or make assumptions beyond the provided information.
4. Maintain the original structure and format of the prompt.

## Output Format
Return only the fully revised prompt. Do not include commentary, summaries, or code fences.
"""

REVISE_USER_PROMPT = f"""
Here is the original prompt:
---
{original_prompt}
---

Here are the issues to fix:
{issues_str}

Please return **only** the fully revised prompt. Do not include commentary, summaries, or explanations.
"""


In [24]:
revised_response = client.responses.create(
    model=MODEL,
    input=REVISE_USER_PROMPT,
    instructions=REVISE_SYSTEM_PROMPT,
    temperature=0.0
)

revised_prompt = revised_response.output_text
print("\n🔄 Revised prompt:\n------------------")
print(revised_response.output_text)


🔄 Revised prompt:
------------------
[System]
Please act as an impartial judge and evaluate the quality of the responses provided by two AI assistants to the user question displayed below. You should choose the assistant that follows the user’s instructions and answers the user’s question better. Your evaluation should be based on the following criteria:

- Helpfulness: The extent to which the response addresses the user’s needs and provides useful information.
- Relevance: How closely the response pertains to the user’s question and instructions.
- Accuracy: The correctness and factual reliability of the information provided.
- Depth: The level of insight, explanation, or reasoning demonstrated in the response.
- Creativity: The originality or resourcefulness shown in addressing the question, where appropriate.
- Level of Detail: The thoroughness and completeness of the response.

All criteria should be considered equally unless the user’s instructions indicate otherwise. 

Begin you

我们来并排对比一下改进前后的提示词，查看高亮显示的修改：

In [25]:
show_critique_and_diff(original_prompt, revised_prompt)

## 第 5 步：评估与迭代
最后，通过以下方式评估改进后的提示词：

- 使用有代表性的评估示例或数据进行测试。

- 分析响应，确保达到预期结果。

- 如需进一步改进，则重复前面的步骤进行迭代。

持续的测试和改进可以确保你的提示词始终达到预期效果。

### 当前示例

我们来评估当前的提示词迁移是否真正提升了这个评审任务的表现。原始提示词取自这篇[论文](https://arxiv.org/pdf/2306.05685)，用于在两个 assistant 的回答之间充当评审。方便的是，该论文提供了一组人工标注的真实标签，因此我们可以衡量 LLM 评审与人类判断一致的频率。

因此，我们的成功指标是：与基线提示词生成的判断相比，迁移后提示词生成的判断与人类评估的吻合程度如何。作为背景，我们使用的基准是 MT-Bench 的一个子集，其中包含多轮对话。在本示例中，我们评估 200 行对话，每一行都比较不同模型对的表现。



在我们的评估子集上，一个有用的参考基准是人与人之间的一致性，因为每段对话都由多名标注者评分。人类对于哪个 assistant 的回答更好并不总能达成一致，因此我们也不会期望评审模型达到完全一致。对于第 1 轮（不含平局），人类之间在 81% 的情况下意见一致；对于第 2 轮，这一比例为 76%。

![模型一致性图 3](../images/prompt_migrator_fig.png)

与迁移前的模型相比，GPT-4（论文中使用的模型）与人类判断的一致率在第 1 轮为 74%，第 2 轮为 71%，表现不错，但仍低于人与人一致性的上限。切换到 GPT-4.1（使用相同提示词）后，一致率有所提升：第 1 轮为 77%，第 2 轮为 72%。最后，在专门针对 GPT-4.1 迁移并调优提示词之后，一致率进一步提高，第 1 轮达到 80%，第 2 轮为 72%，非常接近人类标注者之间的一致水平。

综合来看，我们可以看到，提示词迁移和升级到更强大的模型都提升了示例任务上的一致率。现在就用你的提示词试试吧！

## 第 6 步：（可选）自动应用 GPT-4.1 最佳实践

在这一步中，将自动应用 GPT-4.1 最佳实践来增强你的原始提示词。我们强烈建议你手动检查所做的修改，并决定是否保留。

参考资料请见 [4.1 提示词指南](https://cookbook.openai.com/examples/gpt4-1_prompting_guide)。

In [29]:
BEST_PRACTICES_SYSTEM_PROMPT = """
## Task
Your task is to take a **Baseline Prompt** (provided by the user) and output a **Revised Prompt** that keeps the original wording and order as intact as possible **while surgically inserting improvements that follow the “GPT‑4.1 Best Practices” reference**.

## How to Edit
1. **Keep original text** — Only remove something if it directly goes against a best practice. Otherwise, keep the wording, order, and examples as they are.
2. **Add best practices only when clearly helpful.** If a guideline doesn’t fit the prompt or its use case (e.g., diff‑format guidance on a non‑coding prompt), just leave that part of the prompt unchanged.
3. **Where to add improvements** (use Markdown `#` headings):
   - At the very top, add *Agentic Reminders* (like Persistence, Tool-calling, or Planning) — only if relevant. Don’t add these if the prompt doesn’t require agentic behavior (agentic means prompts that involve planning or running tools for a while).
   - When adding sections, follow this order if possible. If some sections do not make sense, don't add them:
     1. `# Role & Objective`  
        - State who the model is supposed to be (the role) and what its main goal is.
     2. `# Instructions`  
        - List the steps, rules, or actions the model should follow to complete the task.
     3. *(Any sub-sections)*  
        - Include any extra sections such as sub-instructions, notes or guidelines already in the prompt that don’t fit into the main categories.
     4. `# Reasoning Steps`  
        - Explain the step-by-step thinking or logic the model should use when working through the task.
     5. `# Output Format`  
        - Describe exactly how the answer should be structured or formatted (e.g., what sections to include, how to label things, or what style to use).
     6. `# Examples`  
        - Provide sample questions and answers or sample outputs to show the model what a good response looks like.
     7. `# Context`  
        - Supply any background information, retrieved context, or extra details that help the model understand the task better.
   - Don’t introduce new sections that don’t exist in the Baseline Prompt. For example, if there’s no `# Examples` or no `# Context` section, don’t add one.
4. If the prompt is for long context analysis or long tool use, repeat key Agentic Reminders, Important Reminders and Output Format points at the end.
5. If there are class labels, evaluation criterias or key concepts, add a definition to each to define them concretely.
5. Add a chain-of-thought trigger at the end of main instructions (like “Think step by step...”), unless one is already there or it would be repetitive.
6. For prompts involving tools or sample phrases, add Failure-mode bullets:
   - “If you don’t have enough info to use a tool, ask the user first.”
   - “Vary sample phrases to avoid repetition.”
7. Match the original tone (formal or casual) in anything you add.
8. **Only output the full Revised Prompt** — no explanations, comments, or diffs. Do not output "keep the original...", you need to fully output the prompt, no shortcuts.
9. Do not delete any sections or parts that are useful and add value to the prompt and doesn't go against the best practices.
10. **Self-check before sending:** Make sure there are no typos, duplicated lines, missing headings, or missed steps.


## GPT‑4.1 Best Practices Reference  
1. **Persistence reminder**: Explicitly instructs the model to continue working until the user's request is fully resolved, ensuring the model does not stop early.
2. **Tool‑calling reminder**: Clearly tells the model to use available tools or functions instead of making assumptions or guesses, which reduces hallucinations.
3. **Planning reminder**: Directs the model to create a step‑by‑step plan and reflect before and after tool calls, leading to more accurate and thoughtful output.
4. **Scaffold structure**: Requires a consistent and predictable heading order (e.g., Role, Instructions, Output Format) to make prompts easier to maintain.
5. **Instruction placement (long context)**: Ensures that key instructions are duplicated or placed strategically so they remain visible and effective in very long prompts.
6. **Chain‑of‑thought trigger**: Adds a phrase that encourages the model to reason step by step, which improves logical and thorough responses.
7. **Instruction‑conflict hygiene**: Checks for and removes any contradictory instructions, ensuring that the most recent or relevant rule takes precedence.
8. **Failure‑mode mitigations**: Adds safeguards against common errors, such as making empty tool calls or repeating phrases, to improve reliability.
9. **Diff / code‑edit format**: Specifies a robust, line‑number‑free diff or code‑edit style for output, making changes clear and easy to apply.
10. **Label Definitions**: Defines all the key labels or terms that are used in the prompt so that the model knows what they mean.
"""

In [30]:
best_practices_response = client.responses.create(
    model="o3",
    input="BASELINE_PROMPT: " + revised_prompt,
    instructions=BEST_PRACTICES_SYSTEM_PROMPT,
    reasoning={"effort": "high"}
)

improved_prompt = best_practices_response.output_text
print("\nImproved prompt:\n")
print(improved_prompt)


Improved prompt:

# Role & Objective
You are an impartial judge. Your goal is to determine which of two AI assistant answers better fulfills the user’s request.

# Instructions  
Follow the steps below exactly and remain strictly neutral:

1. Read the User Question and both assistant answers in full.  
2. Evaluate each answer against **all** six criteria, treating them with equal weight unless the user explicitly states otherwise:
   • Helpfulness – Does the response address the user’s needs and provide useful information?  
   • Relevance – How closely does the response pertain to the user’s question and instructions?  
   • Accuracy – Is the information correct and factually reliable?  
   • Depth – Does the answer show insight, explanation, or reasoning?  
   • Creativity – Is the approach original or resourceful when appropriate?  
   • Level of Detail – Is the response thorough and complete?  
3. Stay impartial:  
   • Ignore the order in which the answers appear.  
   • Ignore t

In [31]:
show_critique_and_diff(revised_prompt, improved_prompt)